In [ ]:
import os
import pandas as pd

project_path = os.path.dirname(__file__)
implied_odds_file_path = os.path.join(project_path, "nba_custom_implied_odds.csv")
odds_df=pd.read_csv(implied_odds_file_path)
odds_df.head(10)

,person_id,first_name,last_name,series_id,series_game_number,game_id,game_datetime_est,win,implied_win_prob,game_residual,cumulative_series_residual,player_series_order
0,76001,Alaa,Abdelnaby,1991westconffinals16106127471610612757,3,49000058,1991-05-24 20:00:00,0,NaN,NaN,NaN,1
1,76001,Alaa,Abdelnaby,1991westconffinals16106127471610612757,4,49000060,1991-05-26 20:00:00,0,NaN,NaN,NaN,1
2,76001,Alaa,Abdelnaby,1991westconfsemifinals16106127571610612762,1,49000037,1991-05-07 20:00:00,1,NaN,NaN,NaN,2
3,76001,Alaa,Abdelnaby,1991westfirstround16106127571610612760,2,49000016,1991-04-28 20:00:00,1,NaN,NaN,NaN,3
4,76001,Alaa,Abdelnaby,1991westfirstround16106127571610612760,5,49000031,1991-05-04 20:00:00,1,NaN,NaN,NaN,3
5,76001,Alaa,Abdelnaby,1992nbafinals16106127411610612757,6,49103001,1992-06-03 19:00:00,0,NaN,NaN,NaN,4
6,76001,Alaa,Abdelnaby,1992westconffinals16106127571610612762,1,49100054,1992-05-16 20:00:00,1,NaN,NaN,NaN,5
7,76001,Alaa,Abdelnaby,1992westconffinals16106127571610612762,2,49100058,1992-05-19 20:00:00,1,NaN,NaN,NaN,5
8,76001,Alaa,Abdelnaby,1992westconffinals16106127571610612762,4,49100062,1992-05-24 20:00:00,0,NaN,NaN,NaN,5
9,76001,Alaa,Abdelnaby,1992westconfsemifinals16106127561610612757,4,49100046,1992-05-11 20:00:00,1,NaN,NaN,NaN,6


In [5]:
odds_non_null_df = (
    odds_df
    .dropna(
        subset=[
            "implied_win_prob",
            "game_residual",
            "cumulative_series_residual"
        ]
    )
    .assign(
        player_max_cumulative_series_residual=lambda df: (
            df.groupby("person_id")["cumulative_series_residual"].transform("max")
        )
    )
    .sort_values(
        by=["player_max_cumulative_series_residual", "game_datetime_est"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

odds_non_null_df
# odds_non_null_df.to_csv("odds_non_null_results.csv", index=False)

,person_id,first_name,last_name,series_id,series_game_number,game_id,game_datetime_est,win,implied_win_prob,game_residual,cumulative_series_residual,player_series_order,player_max_cumulative_series_residual
0,2200,Pau,Gasol,2018westfirstround16106127441610612759,5,41700155,2018-04-24 22:30:00,0,0.5208,-0.5208,-1.6174,26,2.189
1,2200,Pau,Gasol,2018westfirstround16106127441610612759,4,41700154,2018-04-22 15:30:00,1,0.5222,0.4778,-1.0966,26,2.189
2,2200,Pau,Gasol,2018westfirstround16106127441610612759,3,41700153,2018-04-19 21:30:00,0,0.5258,-0.5258,-1.5744,26,2.189
3,2200,Pau,Gasol,2018westfirstround16106127441610612759,2,41700152,2018-04-16 22:30:00,0,0.5245,-0.5245,-1.0485,26,2.189
4,2200,Pau,Gasol,2018westfirstround16106127441610612759,1,41700151,2018-04-14 15:00:00,0,0.5240,-0.5240,-0.5240,26,2.189
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18897,1628401,Derrick,White,2018westfirstround16106127441610612759,3,41700153,2018-04-19 21:30:00,0,0.5258,-0.5258,-1.5744,1,-0.524
18898,1628401,Derrick,White,2018westfirstround16106127441610612759,2,41700152,2018-04-16 22:30:00,0,0.5245,-0.5245,-1.0485,1,-0.524
18899,1628401,Derrick,White,2018westfirstround16106127441610612759,1,41700151,2018-04-14 15:00:00,0,0.5240,-0.5240,-0.5240,1,-0.524
18900,133,David,Wesley,2007nbafinals16106127391610612759,4,40600404,2007-06-14 21:00:00,0,0.5293,-0.5293,-1.0563,12,-0.527


In [25]:
leading_players_residual_df = (
    odds_df
    .dropna(
        subset=[
            "implied_win_prob",
            "game_residual",
            "cumulative_series_residual",
            "game_datetime_est"
        ]
    )
    .assign(
        game_datetime_est=lambda df: pd.to_datetime(df["game_datetime_est"]),
        player_max_cumulative_series_residual=lambda df: (
            df.groupby("person_id")["cumulative_series_residual"].transform("max")
        )
    )
    .assign(
        player_residual_rank=lambda df: (
            df["player_max_cumulative_series_residual"]
            .rank(method="dense", ascending=False)
            .astype(int)
        )
    )
    .sort_values(
        by=[
            "player_residual_rank",
            "game_datetime_est"
        ],
        ascending=[
            True,
            True
        ]
    )
    .reset_index(drop=True)
)

leading_players_residual_df

,person_id,first_name,last_name,series_id,series_game_number,game_id,game_datetime_est,win,implied_win_prob,game_residual,cumulative_series_residual,player_series_order,player_max_cumulative_series_residual,player_residual_rank
0,965,Derek,Fisher,2007westfirstround16106127451610612762,1,40600171,2007-04-21 21:30:00,0,0.5224,-0.5224,-0.5224,28,2.189,1
1,1956,Ira,Newble,2007eastfirstround16106127391610612764,1,40600111,2007-04-22 12:30:00,1,0.5229,0.4771,0.4771,5,2.189,1
2,977,Kobe,Bryant,2007westfirstround16106127471610612756,1,40600151,2007-04-22 15:00:00,0,0.3969,-0.3969,-0.3969,27,2.189,1
3,200770,Jordan,Farmar,2007westfirstround16106127471610612756,1,40600151,2007-04-22 15:00:00,0,0.3969,-0.3969,-0.3969,1,2.189,1
4,1885,Lamar,Odom,2007westfirstround16106127471610612756,1,40600151,2007-04-22 15:00:00,0,0.3969,-0.3969,-0.3969,4,2.189,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18897,1628401,Derrick,White,2018westfirstround16106127441610612759,3,41700153,2018-04-19 21:30:00,0,0.5258,-0.5258,-1.5744,1,-0.524,117
18898,1628401,Derrick,White,2018westfirstround16106127441610612759,4,41700154,2018-04-22 15:30:00,1,0.5222,0.4778,-1.0966,1,-0.524,117
18899,1628401,Derrick,White,2018westfirstround16106127441610612759,5,41700155,2018-04-24 22:30:00,0,0.5208,-0.5208,-1.6174,1,-0.524,117
18900,133,David,Wesley,2007nbafinals16106127391610612759,3,40600403,2007-06-12 21:00:00,0,0.5270,-0.5270,-0.5270,12,-0.527,118


In [3]:
person_summary_df = (
    odds_df
    .groupby(["person_id", "first_name", "last_name"], as_index=False)
    .agg(
        sum_of_implied_win_prob=("implied_win_prob","sum"),
        sum_of_game_residual=("game_residual","sum"),
        # sum_of_cumulative_series_residual=("cumulative_series_residual",""),
        distinct_series_id=("series_id", "nunique"),
        distinct_game_id=("game_id", "nunique"),
        total_wins=("win", "sum"),
        total_rows=("person_id", "size"),
    )
    .sort_values(by="sum_of_game_residual", ascending=False)
)

person_summary_df

,person_id,first_name,last_name,sum_of_implied_win_prob,sum_of_game_residual,distinct_series_id,distinct_game_id,total_wins,total_rows
364,2544,LeBron,James,114.7892,33.2108,57,299,188,299
388,2592,James,Jones,85.7934,26.2066,37,201,128,201
829,202691,Klay,Thompson,52.3814,19.6186,25,137,92,137
853,203110,Draymond,Green,52.3814,19.6186,27,149,97,149
394,2733,Shaun,Livingston,48.7651,19.2349,24,129,89,129
...,...,...,...,...,...,...,...,...,...
749,200794,Paul,Millsap,44.0635,-8.0635,23,130,57,130
811,202323,Evan,Turner,27.2661,-8.2661,13,69,27,69
721,101109,Raymond,Felton,22.7812,-8.7812,9,49,15,49
382,2581,Steve,Blake,25.4454,-9.4454,12,60,20,60
